# Wafer Defect Pattern Classification
**Dataset:** WM-811K (811,457 real wafer maps · 8 defect classes)  
**Author:** Vigneshwari Jayaprakash | M.S. Data Science, ASU 4.0 GPA  
**GitHub:** [github.com/VigneshwariJayaprakash](https://github.com/VigneshwariJayaprakash)

---

## Project Overview
End-to-end wafer defect classification pipeline directly applicable to Intel's **ADC (Automated Defect Classification)** and **yield analysis** workflows:

| Stage | What it does |
|---|---|
| **1. Data Pipeline** | Load + preprocess real WM-811K wafer maps |
| **2. EDA** | Class distribution, defect density, spatial analysis |
| **3. CNN Feature Extractor** | PyTorch ResNet-based encoder for wafer map embeddings |
| **4. Ensemble Classifier** | XGBoost + Random Forest on CNN features |
| **5. SHAP Explainability** | Per-class feature attribution for process engineers |
| **6. Yield Impact Dashboard** | Defect → root cause → yield estimate → priority |
| **7. Model Export** | Save for FastAPI production serving |

## Defect Classes
| Class | Root Cause | Yield Impact |
|---|---|---|
| Center | CMP non-uniformity | Medium |
| Donut | Photoresist edge bead | Medium |
| Edge-Ring | Edge exclusion failure | High |
| Edge-Loc | Chuck mark / handling | Medium |
| Loc | Particle contamination | Medium |
| Near-Full | Process excursion | Critical |
| Scratch | Mechanical damage | High |
| Random | Baseline process variation | Low |

## 0. Setup & Installation

In [ ]:
# Install dependencies (run once)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# !pip install scikit-learn xgboost shap pandas numpy matplotlib seaborn scipy fastapi uvicorn joblib
# !pip install kaggle  # for dataset download

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy.io import loadmat
from pathlib import Path
import pickle, joblib, json, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score, roc_auc_score)
import xgboost as xgb
import shap

np.random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DEFECT_CLASSES = ['Center', 'Donut', 'Edge-Ring', 'Edge-Loc',
                  'Loc', 'Near-Full', 'Scratch', 'Random']
N_CLASSES = len(DEFECT_CLASSES)
PALETTE = ['#003087','#0068B5','#00C7FD','#76D6FF',
           '#FFB347','#FF6B35','#E63946','#8B4513']

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'Classes: {DEFECT_CLASSES}')

## 1. Data Pipeline — WM-811K Dataset

**Download instructions:**
```bash
# Option A: Kaggle CLI
kaggle datasets download -d qingyi/wm811k-wafer-map
unzip wm811k-wafer-map.zip -d data/

# Option B: Direct from paper authors
# https://www.kaggle.com/datasets/qingyi/wm811k-wafer-map
```
The dataset is a `.pkl` file containing a DataFrame with columns:
- `waferMap`: 2D numpy array (binary die map)
- `failureType`: defect class label
- `waferIndex`, `lotName`, `waferSize`, `dieSize`

In [ ]:
# ── Cell 1: Load WM-811K Dataset ──────────────────────────────────────

DATA_PATH = Path('../data/LSWMD.pkl')  # adjust path as needed

def load_wm811k(path: Path) -> pd.DataFrame:
    """Load and clean the WM-811K dataset."""
    print(f'Loading WM-811K from {path}...')
    df = pd.read_pickle(path)
    print(f'Raw shape: {df.shape}')
    print(f'Columns: {df.columns.tolist()}')

    # Normalize failure type labels
    df['failureType'] = df['failureType'].apply(
        lambda x: x[0][0] if isinstance(x, np.ndarray) else x
    )

    # Keep only labeled samples
    labeled = df[df['failureType'].isin(DEFECT_CLASSES)].copy()
    labeled = labeled.reset_index(drop=True)
    print(f'Labeled samples: {len(labeled):,}')
    return labeled

df = load_wm811k(DATA_PATH)
print('\nClass distribution:')
print(df['failureType'].value_counts())

In [ ]:
# ── Cell 2: Preprocessing — Resize & Normalize Wafer Maps ─────────────

TARGET_SIZE = (64, 64)  # resize all maps to 64x64 for CNN input

def preprocess_wafer_map(wm: np.ndarray, size=TARGET_SIZE) -> np.ndarray:
    """
    Preprocess a raw wafer map:
    1. Resize to fixed size
    2. Normalize to [0, 1]
    3. Create circular wafer mask
    """
    from scipy.ndimage import zoom

    # Handle ragged sizes across wafer lots
    h, w = wm.shape
    zoom_h = size[0] / h
    zoom_w = size[1] / w
    resized = zoom(wm.astype(float), (zoom_h, zoom_w), order=1)

    # Clip to [0,1] (original values: 0=pass, 1=fail, 2=untested)
    # Map: 0→0 (good die), 1→1 (defective die), 2→0.5 (edge/untested)
    resized = np.clip(resized, 0, 1)
    return resized.astype(np.float32)

print('Preprocessing wafer maps...')
df['processed_map'] = df['waferMap'].apply(preprocess_wafer_map)

# Stack into array
X_maps = np.stack(df['processed_map'].values)  # (N, 64, 64)
y_labels = df['failureType'].values

le = LabelEncoder().fit(DEFECT_CLASSES)
y = le.transform(y_labels)

print(f'X shape: {X_maps.shape}')
print(f'y shape: {y.shape}')
print(f'Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

## 2. Exploratory Data Analysis

In [ ]:
# ── Cell 3: EDA — Class Distribution & Defect Patterns ────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('WM-811K Dataset — Class Distribution', fontsize=13, fontweight='bold')

counts = pd.Series(y_labels).value_counts().reindex(DEFECT_CLASSES)
bars = axes[0].bar(DEFECT_CLASSES, counts.values, color=PALETTE, alpha=0.85, edgecolor='white')
axes[0].set_title('Sample Count per Defect Class', fontweight='bold')
axes[0].set_ylabel('Count'); axes[0].tick_params(axis='x', rotation=35)
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 50,
                 f'{v:,}', ha='center', fontsize=8, fontweight='bold')
axes[0].set_yscale('log')  # log scale due to heavy imbalance

axes[1].pie(counts.values, labels=DEFECT_CLASSES, colors=PALETTE,
            autopct='%1.1f%%', startangle=90, pctdistance=0.75)
axes[1].set_title(f'Class Proportion\n(Imbalance ratio: {counts.max()//counts.min()}x)', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total labeled samples : {len(y_labels):,}')
print(f'Class imbalance ratio : {counts.max()//counts.min()}x')
print(f'Majority class        : {counts.idxmax()} ({counts.max():,})')
print(f'Minority class        : {counts.idxmin()} ({counts.min():,})')

In [ ]:
# ── Cell 4: EDA — Visualize One Example of Each Defect Type ───────────

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('WM-811K — Real Wafer Map Samples (One Per Class)',
             fontsize=13, fontweight='bold')

for idx, (cls, ax) in enumerate(zip(DEFECT_CLASSES, axes.flat)):
    # Pick first example of each class
    mask = y_labels == cls
    sample = X_maps[mask][0]

    ax.imshow(sample, cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(cls, fontsize=11, fontweight='bold', color=PALETTE[idx])
    ax.set_xticks([]); ax.set_yticks([])

    defect_rate = sample[sample > 0.4].sum() / (sample >= 0).sum()
    ax.set_xlabel(f'Defect density: {defect_rate:.1%}', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('../figures/wafer_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 5: EDA — Spatial Defect Density Heatmaps ─────────────────────
# Average defect map per class reveals the spatial signature of each failure

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Average Defect Density Map per Class\n(Spatial signature of each failure mode)',
             fontsize=12, fontweight='bold')

for idx, (cls, ax) in enumerate(zip(DEFECT_CLASSES, axes.flat)):
    mask = y_labels == cls
    avg_map = X_maps[mask].mean(axis=0)  # average across all wafers of this class

    im = ax.imshow(avg_map, cmap='hot', vmin=0, vmax=avg_map.max(),
                   interpolation='bilinear')
    ax.set_title(f'{cls}\n(n={mask.sum():,})', fontsize=9,
                 fontweight='bold', color=PALETTE[idx])
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('../figures/spatial_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Spatial heatmaps show clearly distinct defect signatures per class.')
print('→ Edge-Ring concentrates at wafer edge; Center peaks at center; Scratch shows a linear band.')

## 3. CNN Feature Extractor (PyTorch)
We use a **lightweight CNN** with residual connections, pre-trained on wafer map structure, to extract a 128-dimensional embedding from each wafer map. These embeddings are then fed into the ensemble classifier.

In [ ]:
# ── Cell 6: WaferMap Dataset class ────────────────────────────────────

class WaferMapDataset(Dataset):
    """
    PyTorch Dataset for wafer maps.
    Converts 2D maps to 3-channel tensors (repeat grayscale → RGB)
    so we can use pretrained CNN backbones.
    """
    def __init__(self, maps: np.ndarray, labels: np.ndarray,
                 augment: bool = False):
        self.maps = maps
        self.labels = labels
        self.augment = augment

        base_transforms = [
            transforms.ToPILImage(),
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ]
        if augment:
            base_transforms.insert(1, transforms.RandomHorizontalFlip())
            base_transforms.insert(2, transforms.RandomVerticalFlip())
            base_transforms.insert(3, transforms.RandomRotation(90))

        self.transform = transforms.Compose(base_transforms)

    def __len__(self):
        return len(self.maps)

    def __getitem__(self, idx):
        wm = self.maps[idx]  # (64, 64) float32
        # Convert to uint8 for PIL
        wm_uint8 = (wm * 255).astype(np.uint8)
        tensor = self.transform(wm_uint8)  # (1, 64, 64)
        # Repeat to 3 channels for pretrained CNN compatibility
        tensor = tensor.repeat(3, 1, 1)    # (3, 64, 64)
        return tensor, self.labels[idx]

In [ ]:
# ── Cell 7: CNN Feature Extractor Architecture ─────────────────────────

class WaferCNNEncoder(nn.Module):
    """
    Lightweight CNN encoder for wafer map feature extraction.
    Architecture:
      - 3 convolutional blocks with BatchNorm + ReLU + MaxPool
      - Residual skip connections for gradient flow
      - Global average pooling → 128-dim embedding

    Designed for wafer maps (64x64, binary-ish patterns).
    Smaller and faster than ResNet for this domain.
    """
    def __init__(self, embedding_dim: int = 128):
        super().__init__()

        # Block 1: 3 → 32 channels
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)  # 64→32
        )

        # Block 2: 32 → 64 channels
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)  # 32→16
        )
        self.skip1 = nn.Conv2d(32, 64, kernel_size=1, stride=2)

        # Block 3: 64 → 128 channels
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)  # 16→8
        )
        self.skip2 = nn.Conv2d(64, 128, kernel_size=1, stride=2)

        # Global average pooling → embedding
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        # Classification head (for supervised pre-training)
        self.classifier = nn.Linear(embedding_dim, N_CLASSES)

    def forward(self, x, return_embedding: bool = False):
        # Block 1
        x1 = self.conv1(x)

        # Block 2 with residual
        x2 = self.conv2(x1) + self.skip1(x1)

        # Block 3 with residual
        x3 = self.conv3(x2) + self.skip2(x2)

        # GAP → embedding
        pooled = self.gap(x3)
        emb = self.embedding(pooled)

        if return_embedding:
            return emb
        return self.classifier(emb)


# Instantiate
model = WaferCNNEncoder(embedding_dim=128).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')
print(model)

In [ ]:
# ── Cell 8: CNN Training ───────────────────────────────────────────────

# Train/val split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_maps, y, test_size=0.15, random_state=42, stratify=y
)

# Class weights for imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)

# Datasets & loaders
train_ds = WaferMapDataset(X_tr, y_tr, augment=True)
val_ds   = WaferMapDataset(X_val, y_val, augment=False)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=2)

# Loss + optimizer
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += len(y_batch)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * len(y_batch)
            correct += (logits.argmax(1) == y_batch).sum().item()
            total += len(y_batch)
    return total_loss / total, correct / total

# Training loop
EPOCHS = 25
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), '../models/cnn_encoder_best.pt')

    if epoch % 5 == 0:
        print(f'Epoch {epoch:2d}/{EPOCHS} | '
              f'Train loss: {tr_loss:.4f} acc: {tr_acc:.3f} | '
              f'Val loss: {vl_loss:.4f} acc: {vl_acc:.3f}')

print(f'\nBest val accuracy: {best_val_acc:.3f}')

In [ ]:
# ── Cell 9: Training Curves ────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, EPOCHS + 1)

ax1.plot(epochs, history['train_loss'], label='Train', color='#003087', linewidth=2)
ax1.plot(epochs, history['val_loss'],   label='Val',   color='#E63946', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history['train_acc'], label='Train', color='#003087', linewidth=2)
ax2.plot(epochs, history['val_acc'],   label='Val',   color='#E63946', linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Ensemble Classifier (CNN Embeddings + XGBoost + Random Forest)

In [ ]:
# ── Cell 10: Extract CNN Embeddings ───────────────────────────────────

# Load best CNN weights
model.load_state_dict(torch.load('../models/cnn_encoder_best.pt', map_location=DEVICE))
model.eval()

def extract_embeddings(model, maps: np.ndarray,
                        batch_size: int = 256) -> np.ndarray:
    """Extract 128-dim CNN embeddings for all wafer maps."""
    dataset = WaferMapDataset(maps, np.zeros(len(maps), dtype=int))
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=False, num_workers=2)
    all_embs = []
    with torch.no_grad():
        for X_batch, _ in loader:
            emb = model(X_batch.to(DEVICE), return_embedding=True)
            all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)

print('Extracting CNN embeddings...')
X_emb = extract_embeddings(model, X_maps)
print(f'Embeddings shape: {X_emb.shape}')  # (N, 128)

# Also extract domain-engineered features
def extract_domain_features(wm: np.ndarray) -> np.ndarray:
    """14 hand-crafted spatial features from wafer domain knowledge."""
    size = wm.shape[0]
    cx, cy = size // 2, size // 2
    radius = size // 2

    valid = np.array([[np.sqrt((i-cx)**2 + (j-cy)**2) <= radius
                       for j in range(size)] for i in range(size)])
    defective = wm > 0.4
    n_valid  = valid.sum()
    n_defect = (defective & valid).sum()
    density  = n_defect / n_valid if n_valid > 0 else 0

    # Radial zone densities
    zones = []
    for r0, r1 in [(0, 0.33), (0.33, 0.67), (0.67, 1.0)]:
        zone = np.array([[r0*radius < np.sqrt((i-cx)**2+(j-cy)**2) <= r1*radius
                          for j in range(size)] for i in range(size)])
        zv = (zone & valid).sum()
        zd = (zone & valid & defective).sum()
        zones.append(zd / zv if zv > 0 else 0)

    # Quadrant densities
    quads = []
    for qi, qj in [(slice(None,cx), slice(None,cy)), (slice(None,cx), slice(cy,None)),
                   (slice(cx,None), slice(None,cy)), (slice(cx,None), slice(cy,None))]:
        qv = valid[qi,qj].sum(); qd = (defective & valid)[qi,qj].sum()
        quads.append(qd / qv if qv > 0 else 0)

    row_vars = np.var(wm * valid, axis=1)
    col_vars = np.var(wm * valid, axis=0)
    def_pos  = np.argwhere(defective & valid)
    compactness = 1/(1 + np.mean(np.std(def_pos, axis=0))) if len(def_pos) > 1 else 0

    return np.array([
        density, *zones,
        np.std(quads), np.max(quads),
        zones[0] - zones[2],              # radial gradient
        max(row_vars.max(), col_vars.max()), # linearity score
        zones[1] - (zones[0]+zones[2])/2, # ring score
        compactness, *quads
    ])

print('Extracting domain features...')
X_domain = np.array([extract_domain_features(wm) for wm in X_maps])
print(f'Domain features shape: {X_domain.shape}')  # (N, 14)

# Combine CNN embeddings + domain features
X_combined = np.hstack([X_emb, X_domain])
print(f'Combined features: {X_combined.shape}')  # (N, 142)

In [ ]:
# ── Cell 11: Train Ensemble Classifier ────────────────────────────────

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler().fit(X_tr2)
X_tr2s = scaler.transform(X_tr2)
X_te2s = scaler.transform(X_te2)

# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=42, n_jobs=-1
)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=15,
    class_weight='balanced', random_state=42, n_jobs=-1
)

# Soft voting ensemble
ensemble = VotingClassifier(
    estimators=[('xgb', xgb_model), ('rf', rf_model)],
    voting='soft'
)

print('Training ensemble (XGBoost + Random Forest on CNN embeddings)...')
ensemble.fit(X_tr2s, y_tr2)

y_pred = ensemble.predict(X_te2s)
acc = accuracy_score(y_te2, y_pred)
f1_w = f1_score(y_te2, y_pred, average='weighted')
f1_m = f1_score(y_te2, y_pred, average='macro')

print(f'\nEnsemble Results:')
print(f'  Accuracy      : {acc:.4f}')
print(f'  Weighted F1   : {f1_w:.4f}')
print(f'  Macro F1      : {f1_m:.4f}')
print(f'\n{classification_report(y_te2, y_pred, target_names=DEFECT_CLASSES)}')

# Save model artifacts
joblib.dump(ensemble, '../models/ensemble_classifier.pkl')
joblib.dump(scaler,   '../models/feature_scaler.pkl')
joblib.dump(le,       '../models/label_encoder.pkl')
print('Models saved to ../models/')

In [ ]:
# ── Cell 12: Confusion Matrix ──────────────────────────────────────────

cm = confusion_matrix(y_te2, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=DEFECT_CLASSES, yticklabels=DEFECT_CLASSES,
            ax=ax1, linewidths=0.5)
ax1.set_title('Confusion Matrix (Raw Counts)', fontweight='bold', fontsize=12)
ax1.set_ylabel('True'); ax1.set_xlabel('Predicted')
ax1.tick_params(axis='x', rotation=35)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=DEFECT_CLASSES, yticklabels=DEFECT_CLASSES,
            ax=ax2, linewidths=0.5, vmin=0, vmax=1)
ax2.set_title('Confusion Matrix (Normalized)', fontweight='bold', fontsize=12)
ax2.set_ylabel('True'); ax2.set_xlabel('Predicted')
ax2.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig('../figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. SHAP Explainability — Per Defect Class

In [ ]:
# ── Cell 13: SHAP Feature Importance ──────────────────────────────────

DOMAIN_FEATURE_NAMES = [
    'defect_density', 'zone_center', 'zone_mid', 'zone_edge',
    'quadrant_std', 'quadrant_max', 'radial_gradient',
    'linearity_score', 'ring_score', 'cluster_compactness',
    'q0_density', 'q1_density', 'q2_density', 'q3_density'
]
CNN_FEATURE_NAMES = [f'cnn_emb_{i}' for i in range(128)]
ALL_FEATURE_NAMES = CNN_FEATURE_NAMES + DOMAIN_FEATURE_NAMES

# Use XGBoost for SHAP (native support)
print('Computing SHAP values (this may take a few minutes)...')
xgb_only = xgb_model.fit(X_tr2s, y_tr2)

# Use a background sample for speed
background = X_te2s[:200]
explainer = shap.TreeExplainer(xgb_only)
shap_values = explainer.shap_values(background)  # list of arrays, one per class

print(f'SHAP values shape: {len(shap_values)} classes x {background.shape}')

# ── Plot: SHAP summary for domain features only (interpretable) ──
# Focus on last 14 features (domain-engineered)
domain_shap = [sv[:, -14:] for sv in shap_values]
domain_bg   = background[:, -14:]

fig, axes = plt.subplots(2, 4, figsize=(16, 10))
fig.suptitle('SHAP Feature Importance — Domain Features per Defect Class\n'
             '(Red = increases defect probability, Blue = decreases)',
             fontsize=12, fontweight='bold')

for idx, (cls, ax) in enumerate(zip(DEFECT_CLASSES, axes.flat)):
    sv = domain_shap[idx]  # (n_samples, 14)
    mean_abs = np.abs(sv).mean(axis=0)
    sorted_idx = np.argsort(mean_abs)[::-1][:8]  # top 8 features

    feat_names = [DOMAIN_FEATURE_NAMES[i] for i in sorted_idx]
    feat_vals  = mean_abs[sorted_idx]

    colors = [PALETTE[idx]] * len(feat_names)
    ax.barh(feat_names[::-1], feat_vals[::-1], color=colors, alpha=0.8)
    ax.set_title(cls, fontsize=10, fontweight='bold', color=PALETTE[idx])
    ax.set_xlabel('Mean |SHAP|', fontsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../figures/shap_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top driver per class
print('\nTop SHAP driver per defect class:')
print('-' * 40)
for idx, cls in enumerate(DEFECT_CLASSES):
    mean_abs = np.abs(domain_shap[idx]).mean(axis=0)
    top_feat = DOMAIN_FEATURE_NAMES[mean_abs.argmax()]
    print(f'  {cls:<12} → {top_feat}')

## 6. Yield Impact & Root Cause Dashboard

In [ ]:
# ── Cell 14: Yield Impact Analysis ────────────────────────────────────

ROOT_CAUSES = {
    'Center'   : 'CMP non-uniformity / spin-coat dispense',
    'Donut'    : 'Photoresist edge bead / thermal gradient',
    'Edge-Ring': 'Edge exclusion / edge bead removal failure',
    'Edge-Loc' : 'Chuck mark / wafer handling damage',
    'Loc'      : 'Particle contamination / local hot spot',
    'Near-Full': 'Process excursion / equipment failure',
    'Scratch'  : 'Mechanical damage / robot arm contact',
    'Random'   : 'Baseline process variation (normal)'
}
PRIORITY = {
    'Center': 'HIGH', 'Donut': 'MEDIUM', 'Edge-Ring': 'HIGH',
    'Edge-Loc': 'MEDIUM', 'Loc': 'MEDIUM', 'Near-Full': 'CRITICAL',
    'Scratch': 'HIGH', 'Random': 'LOW'
}
PRIORITY_COLORS = {
    'CRITICAL': '#E63946', 'HIGH': '#FF6B35', 'MEDIUM': '#FFB347', 'LOW': '#76D6FF'
}

# Compute per-class defect density stats from real data
class_stats = {}
for cls in DEFECT_CLASSES:
    mask = y_labels == cls
    class_maps = X_maps[mask]
    densities = []
    for wm in class_maps[:500]:  # sample 500 for speed
        cx, cy = wm.shape[0]//2, wm.shape[1]//2
        r = min(cx, cy)
        valid = sum(1 for i in range(wm.shape[0]) for j in range(wm.shape[1])
                    if np.sqrt((i-cx)**2+(j-cy)**2) <= r)
        densities.append((wm > 0.4).sum() / valid if valid > 0 else 0)
    mean_d = np.mean(densities)
    class_stats[cls] = {
        'mean_density': mean_d,
        'std_density' : np.std(densities),
        'yield_est'   : max(0, 1 - mean_d),  # simplified Poisson yield
        'n_samples'   : mask.sum()
    }

# ── Dashboard Plot ──
fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# 1. Yield by defect type
ax1 = fig.add_subplot(gs[0, :])
yields = [class_stats[c]['yield_est'] * 100 for c in DEFECT_CLASSES]
bars   = ax1.bar(DEFECT_CLASSES, yields, color=PALETTE, alpha=0.85, edgecolor='white')
ax1.set_ylabel('Estimated Die Yield (%)', fontsize=11)
ax1.set_title('Estimated Die Yield by Defect Pattern',
              fontweight='bold', fontsize=12)
ax1.set_ylim(0, 115)
ax1.axhline(y=95, color='green',  linestyle='--', alpha=0.6, linewidth=1.5, label='95% target')
ax1.axhline(y=80, color='orange', linestyle='--', alpha=0.6, linewidth=1.5, label='80% alert')
ax1.legend(fontsize=9)
for bar, y in zip(bars, yields):
    p = PRIORITY[DEFECT_CLASSES[bars.patches.index(bar)]]
    ax1.text(bar.get_x() + bar.get_width()/2, y + 1,
             f'{y:.1f}%\n[{p}]', ha='center', fontsize=8, fontweight='bold',
             color=PRIORITY_COLORS[p])

# 2. Defect density by class
ax2 = fig.add_subplot(gs[1, 0])
densities_mean = [class_stats[c]['mean_density'] for c in DEFECT_CLASSES]
densities_std  = [class_stats[c]['std_density']  for c in DEFECT_CLASSES]
ax2.barh(DEFECT_CLASSES, densities_mean, xerr=densities_std,
         color=PALETTE, alpha=0.85, capsize=4)
ax2.set_xlabel('Mean Defect Density (± std)')
ax2.set_title('Defect Density per Class', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

# 3. Root cause + priority table
ax3 = fig.add_subplot(gs[1, 1])
ax3.axis('off')
table_data = [[cls, PRIORITY[cls], ROOT_CAUSES[cls][:35] + '...']
              if len(ROOT_CAUSES[cls]) > 35
              else [cls, PRIORITY[cls], ROOT_CAUSES[cls]]
              for cls in DEFECT_CLASSES]
tbl = ax3.table(cellText=table_data,
                colLabels=['Defect', 'Priority', 'Root Cause'],
                cellLoc='left', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#003087')
        cell.set_text_props(color='white', fontweight='bold')
    elif c == 1 and r > 0:
        p = table_data[r-1][1]
        cell.set_facecolor(PRIORITY_COLORS[p])
        cell.set_text_props(fontweight='bold')
    else:
        cell.set_facecolor('#F0F4FA' if r % 2 == 0 else 'white')
    cell.set_edgecolor('#CCCCCC')
ax3.set_title('Defect → Root Cause → Priority', fontweight='bold')

plt.savefig('../figures/yield_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export Model for Production (FastAPI)

In [ ]:
# ── Cell 15: Export Model Artifacts ───────────────────────────────────

# Save CNN encoder as TorchScript for production serving
model.eval()
example_input = torch.randn(1, 3, 64, 64).to(DEVICE)
traced_model = torch.jit.trace(model, example_input)
torch.jit.save(traced_model, '../models/cnn_encoder_traced.pt')

# Save metadata
metadata = {
    'defect_classes'     : DEFECT_CLASSES,
    'root_causes'        : ROOT_CAUSES,
    'priority'           : PRIORITY,
    'input_size'         : [64, 64],
    'embedding_dim'      : 128,
    'n_domain_features'  : 14,
    'total_features'     : 142,
    'model_accuracy'     : float(acc),
    'model_f1_weighted'  : float(f1_w),
}
with open('../models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Exported:')
print('  ../models/cnn_encoder_best.pt      — PyTorch weights')
print('  ../models/cnn_encoder_traced.pt    — TorchScript (production)')
print('  ../models/ensemble_classifier.pkl  — XGBoost + RF ensemble')
print('  ../models/feature_scaler.pkl       — StandardScaler')
print('  ../models/label_encoder.pkl        — LabelEncoder')
print('  ../models/metadata.json            — Model metadata')
print(f'\nFinal model: Accuracy={acc:.1%}  Weighted F1={f1_w:.3f}  Macro F1={f1_m:.3f}')